In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import pandas as pd
import json
import itertools
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Importing our optimizer, models and data loaders
from adasecant import Adasecant 
from models import MLP, CNN, MNIST_ViT, CIFAR_ViT
from dataloaders import get_data_loaders

## Helper functions

In [6]:
# 1. CORE TRAINING & EVALUATION FUNCTIONS

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    start_time = time.time()
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_time = time.time() - start_time
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc, epoch_time

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


# 2. HELPER FUNCTION TO GET OPTIMIZER INSTANCE

def get_optimizer(opt_name, model, params_dict):
    if opt_name == 'SGD':
        return optim.SGD(model.parameters(), **params_dict)
    elif opt_name == 'Adam':
        return optim.Adam(model.parameters(), **params_dict)
    elif opt_name == 'RMSprop':
        return optim.RMSprop(model.parameters(), **params_dict)
    elif opt_name == 'Adasecant':
        return Adasecant(model.parameters(), **params_dict)
    else:
        raise ValueError(f"Unknown optimizer: {opt_name}")

# CNN and MLP experiments (on CIFAR10 and MNIST datasets)

We chose to run these two experiments because these combinations of datasets and models are widely used as baselines for many image recognition tasks. This allowed us to tune the hyperparameters and compare optimizers in a reasonable amount of time, whereas other setups tested in the initial phase of the project were too time-consuming.

## Hyperparameters search

In [3]:
# 1. HYPERPARAMETER GRIDS

PARAM_GRIDS = {
    'SGD': {
        'lr': [0.1, 0.01, 0.001],
        'momentum': [0.9] 
    },
    'Adam': {
        'lr': [0.01, 0.001, 0.0001],
        'eps': [1e-8, 1e-7] 
    },
    'RMSprop': {
        'lr': [0.01, 0.001, 0.0001],
        'eps': [1e-8, 1e-7]
    },
    'Adasecant': {
        'decay': [0.95],
        'gamma_clip': [1.8],
        'use_corrected_grad': [True]
    }
}


# 2. HELPER FUNCTIONS FOR TUNING

def get_subset_loader(loader, max_batches):
    """Yields only a subset of batches to speed up tuning."""
    def subset_generator():
        for i, batch in enumerate(loader):
            if i >= max_batches:
                break
            yield batch
    return subset_generator()


def generate_grid(grid_dict):
    """Converts a dictionary of lists into a list of dictionaries (all combinations)."""
    keys = grid_dict.keys()
    values = grid_dict.values()
    combinations = list(itertools.product(*values))
    return [dict(zip(keys, combo)) for combo in combinations]


# 3. THE TUNING LOOP

def run_tuning():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running tuning on device: {device}")
    
    # TUNING SETUP 
    tasks = [
        {'data': 'mnist', 'model_class': MLP},
        {'data': 'cifar10', 'model_class': CNN}
    ]
    
    batch_sizes = [8, 32, 128, 512]
    
    # Tuning constraints
    TUNE_EPOCHS = 3          # Only train for 3 epochs per combination
    MAX_TRAIN_BATCHES = 50   # Only use ~50 batches per epoch for tuning
    MAX_VAL_BATCHES = 20     # Only use ~20 batches for validation
    
    criterion = nn.CrossEntropyLoss()
    best_hyperparameters = {}

    #  EXECUTION 
    for task in tasks:
        dataset_name = task['data']
        ModelClass = task['model_class']
        best_hyperparameters[dataset_name] = {}
        
        print(f"Starting Tuning: {dataset_name.upper()} with {ModelClass.__name__}")
        
        for batch_size in batch_sizes:
            print(f"\nTuning for Batch Size: {batch_size}\n")
            best_hyperparameters[dataset_name][batch_size] = {}
            
            # Load full loaders once per batch size
            train_loader, test_loader = get_data_loaders(dataset_name, batch_size=batch_size)
            
            for opt_name, grid in PARAM_GRIDS.items():
                print(f"Tuning {opt_name}")
                param_combinations = generate_grid(grid)
                best_loss = float('inf')
                best_params = None
                
                for params in param_combinations:
                    # 1. Initialize a fresh model for this parameter combination
                    model = ModelClass().to(device)
                    optimizer = get_optimizer(opt_name, model, params)
                    
                    val_loss = 0.0
                    
                    # 2. Fast Training Loop
                    for epoch in range(TUNE_EPOCHS):
                        model.train()
                        train_subset = get_subset_loader(train_loader, MAX_TRAIN_BATCHES)
                        for inputs, labels in train_subset:
                            inputs, labels = inputs.to(device), labels.to(device)
                            optimizer.zero_grad()
                            outputs = model(inputs)
                            loss = criterion(outputs, labels)
                            loss.backward()
                            optimizer.step()
                            
                    # 3. Fast Validation Loop
                    model.eval()
                    val_subset = get_subset_loader(test_loader, MAX_VAL_BATCHES)
                    running_val_loss = 0.0
                    total_samples = 0
                    
                    with torch.no_grad():
                        for inputs, labels in val_subset:
                            inputs, labels = inputs.to(device), labels.to(device)
                            outputs = model(inputs)
                            batch_loss = criterion(outputs, labels)
                            running_val_loss += batch_loss.item() * inputs.size(0)
                            total_samples += labels.size(0)
                            
                    val_loss = running_val_loss / max(total_samples, 1)
                    
                    # Log combination result
                    param_str = ", ".join([f"{k}={v}" for k, v in params.items()])
                    print(f"     Params: [{param_str}] -> Val Loss: {val_loss:.4f}")
                    
                    # 4. Check if best
                    if val_loss < best_loss:
                        best_loss = val_loss
                        best_params = params
                        
                print(f"BEST {opt_name} Params for Batch {batch_size}: {best_params} (Loss: {best_loss:.4f})")
                best_hyperparameters[dataset_name][batch_size][opt_name] = best_params

    #  SAVE RESULTS 
    with open("best_hyperparameters.json", "w") as f:
        json.dump(best_hyperparameters, f, indent=4)
        
    print("\nTuning Complete! Results saved to 'best_hyperparameters.json'")

In [ ]:
# Executing the tuning process
run_tuning()

## The experiment loop

In [ ]:
# THE EXPERIMENT

def run_benchmark_experiment():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on device: {device}")
    
    #  EXPERIMENT SETUP 
    # 1. Load Tuned Hyperparameters
    config_file = "best_hyperparameters.json"
        
    with open(config_file, "r") as f:
        best_hyperparameters = json.load(f)
    print(f"Successfully loaded optimized hyperparameters from {config_file}.")

    # 2. Datasets & Models to test
    tasks = [
        {'data': 'mnist', 'model_class': MLP, 'epochs': 15},
        {'data': 'cifar10', 'model_class': CNN, 'epochs': 25}
    ]
    
    # 3. Optimizers
    optimizers_to_test = ['SGD', 'Adam', 'RMSprop', 'Adasecant']
    
    # 4. Batch Sizes
    batch_sizes = [8, 32, 128, 512]
    
    results = []
    criterion = nn.CrossEntropyLoss()

    #  THE EXECUTION LOOP 
    for task in tasks:
        dataset_name = task['data']
        epochs = task['epochs']
        ModelClass = task['model_class']
        
        print(f"Starting Task: {dataset_name.upper()} with {ModelClass.__name__}")
        
        for batch_size in batch_sizes:
            print(f"\n Loading data with Batch Size: {batch_size} ")
            train_loader, test_loader = get_data_loaders(dataset_name, batch_size=batch_size)
            
            for opt_name in optimizers_to_test:
                # Fetch the optimal params for this specific dataset and batch size
                try:
                    tuned_params = best_hyperparameters[dataset_name][str(batch_size)][opt_name]
                except KeyError:
                    print(f"WARNING: Missing config for {dataset_name} | Batch {batch_size} | {opt_name}. Skipping...")
                    continue
                
                param_str = ", ".join([f"{k}={v}" for k, v in tuned_params.items()])
                print(f"Training with {opt_name} (Params: [{param_str}])...")
                
                model = ModelClass().to(device)
                
                try:
                    optimizer = get_optimizer(opt_name, model, tuned_params)
                except Exception as e:
                    print(f"Failed to initialize {opt_name}: {e}")
                    continue
                
                cumulative_time = 0.0
                
                for epoch in range(1, epochs + 1):
                    # Train and Evaluate
                    train_loss, train_acc, epoch_time = train_one_epoch(model, train_loader, criterion, optimizer, device)
                    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
                    
                    cumulative_time += epoch_time
                    
                    # Log the results for this epoch
                    results.append({
                        'Dataset': dataset_name,
                        'Model': ModelClass.__name__,
                        'Batch_Size': batch_size,
                        'Optimizer': opt_name,
                        'Epoch': epoch,
                        'Train_Loss': train_loss,
                        'Train_Acc': train_acc,
                        'Val_Loss': val_loss,
                        'Val_Acc': val_acc,
                        'Epoch_Time_sec': epoch_time,
                        'Cumulative_Time_sec': cumulative_time
                    })
                    
                    # Print progress every 5 epochs
                    if epoch % 5 == 0 or epoch == 1:
                        print(f"  Epoch [{epoch}/{epochs}] | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    #  SAVE RESULTS 
    df_results = pd.DataFrame(results)
    df_results.to_csv("benchmark_experiment_results.csv", index=False)
    print("\nExperiment Complete. Results saved to 'benchmark_experiment_results.csv'")
    return df_results

In [ ]:
# Running the benchmark experiment, the result is a DataFrame
df = run_benchmark_experiment()

## Visualizing the results of CNN and MLP experiments

In [ ]:
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)

def load_data(file_path="benchmark_experiment_results.csv"):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Cannot find {file_path}. Ensure the experiment has finished running.")
    return pd.read_csv(file_path)

def plot_convergence_by_epoch(df, dataset_name, batch_size, save_dir="plots/CNN_MLP"):
    """Plots Training Loss and Val Loss vs Epochs"""
    subset = df[(df['Dataset'] == dataset_name) & (df['Batch_Size'] == batch_size)]
    if subset.empty:
        return
        
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Convergence by Epoch: {dataset_name.upper()} | Batch Size: {batch_size}", fontsize=20, y=1.05)
    
    # Plot 1: Train Loss vs Epoch
    sns.lineplot(data=subset, x='Epoch', y='Train_Loss', hue='Optimizer', marker='o', ax=axes[0], linewidth=2.5)
    axes[0].set_title('Training Loss', fontsize=18)
    axes[0].set_ylabel('Cross Entropy Loss', fontsize=16)
    axes[0].set_xlabel('Epoch', fontsize=16)
    axes[0].legend(fontsize=14, title_fontsize=13)
    axes[0].tick_params(labelsize=14)
    
    # Plot 2: Val Loss vs Epoch
    sns.lineplot(data=subset, x='Epoch', y='Val_Loss', hue='Optimizer', marker='s', ax=axes[1], linewidth=2.5)
    axes[1].set_title('Validation Loss', fontsize=18)
    axes[1].set_ylabel('Loss', fontsize=16)
    axes[1].set_xlabel('Epoch', fontsize=16)
    axes[1].legend(fontsize=14, title_fontsize=13)
    axes[1].tick_params(labelsize=14)
    
    plt.tight_layout()
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(f"{save_dir}/{dataset_name}_b{batch_size}_epochs.png", dpi=300, bbox_inches='tight')
    plt.close()

def plot_convergence_by_time(df, dataset_name, batch_size, save_dir="plots/CNN_MLP"):
    """Plots Training Loss vs Cumulative Wall-clock Time"""
    subset = df[(df['Dataset'] == dataset_name) & (df['Batch_Size'] == batch_size)]
    if subset.empty:
        return

    plt.figure(figsize=(6, 4))
    ax = sns.lineplot(
        data=subset,
        x='Cumulative_Time_sec',
        y='Train_Loss',
        hue='Optimizer',
        marker='X',
        linewidth=2.0,
        markersize=7,
        err_style="bars"
    )

    ax.set_title(
        f"Loss vs. Wall-Clock Time: {dataset_name.upper()} | Batch Size: {batch_size}",
        fontsize=13,
        pad=10
    )
    ax.set_xlabel('Cumulative Training Time (Seconds)', fontsize=11)
    ax.set_ylabel('Training Loss', fontsize=11)

    ax.tick_params(axis='both', which='major', labelsize=9)
    legend = ax.legend(fontsize=9)
    if legend is not None:
        plt.setp(legend.get_title(), fontsize=10)

    plt.tight_layout()
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(f"{save_dir}/{dataset_name}_b{batch_size}_time.png", dpi=600, bbox_inches='tight')
    plt.close()

def plot_overhead_bar_chart(df, save_dir="plots/CNN_MLP"):
    """Plots the average time taken per epoch for each optimizer"""
    avg_time = df.groupby(['Optimizer', 'Dataset'])['Epoch_Time_sec'].mean().reset_index()
    
    plt.figure(figsize=(8, 6))
    sns.barplot(data=avg_time, x='Dataset', y='Epoch_Time_sec', hue='Optimizer')
    
    plt.title("Computational Overhead: Average Seconds per Epoch")
    plt.ylabel('Seconds')
    plt.xlabel('Dataset / Model Architecture')
    
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(f"{save_dir}/computational_overhead.png", dpi=300, bbox_inches='tight')
    plt.close()

def generate_CNN_MLP_visualizations(csv_path="benchmark_experiment_results.csv"):
    print("Loading experiment data...")
    try:
        df = load_data(csv_path)
    except FileNotFoundError as e:
        print(e)
        return

    datasets = df['Dataset'].unique()
    batch_sizes = df['Batch_Size'].unique()
    
    print("Generating plots...")
    for dataset in datasets:
        for batch_size in batch_sizes:
            plot_convergence_by_epoch(df, dataset, batch_size)
            plot_convergence_by_time(df, dataset, batch_size)
            
    plot_overhead_bar_chart(df)
    print("All visualizations saved successfully in the 'plots/CNN_MLP/' directory!")

In [ ]:
# Visualization Execution
generate_CNN_MLP_visualizations()

# Transformer test on MNIST and CIFAR10 datasets

Here we trained a different model architecure - transformer. These types of models have recently got more popular in image recognition tasks.

In [5]:
# THE EXPERIMENT

def run_transformer_experiment():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on device: {device}")
    
    #  EXPERIMENT SETUP 
    # 1. Load Tuned Hyperparameters
    config_file = "best_hyperparameters.json"
        
    with open(config_file, "r") as f:
        best_hyperparameters = json.load(f)
    print(f"Successfully loaded optimized hyperparameters from {config_file}.")

    # 2. Datasets & Models to test (UPDATED FOR TRANSFORMERS)
    tasks = [
        {'data': 'mnist', 'model_class': MNIST_ViT, 'epochs': 15},
        {'data': 'cifar10', 'model_class': CIFAR_ViT, 'epochs': 25}
    ]
    
    # 3. Optimizers to compare 
    optimizers_to_test = ['SGD', 'Adam', 'RMSprop', 'Adasecant']
    
    # 4. Batch Sizes
    batch_sizes = [8, 32, 128, 512]
    
    results = []
    criterion = nn.CrossEntropyLoss()

    #  THE EXECUTION LOOP 
    for task in tasks:
        dataset_name = task['data']
        epochs = task['epochs']
        ModelClass = task['model_class']
        
        print(f"Starting Task: {dataset_name.upper()} with {ModelClass.__name__}")
        
        for batch_size in batch_sizes:
            print(f"\n Loading data with Batch Size: {batch_size} ")
            train_loader, test_loader = get_data_loaders(dataset_name, batch_size=batch_size)
            
            for opt_name in optimizers_to_test:
                try:
                    tuned_params = best_hyperparameters[dataset_name][str(batch_size)][opt_name]
                except KeyError:
                    print(f"WARNING: Missing config for {dataset_name} | Batch {batch_size} | {opt_name}. Skipping...")
                    continue
                
                param_str = ", ".join([f"{k}={v}" for k, v in tuned_params.items()])
                print(f"Training with {opt_name} (Params: [{param_str}])...")
                
                model = ModelClass().to(device)
                
                try:
                    optimizer = get_optimizer(opt_name, model, tuned_params)
                except Exception as e:
                    print(f"Failed to initialize {opt_name}: {e}")
                    continue
                
                cumulative_time = 0.0
                
                for epoch in range(1, epochs + 1):
                    # Train and Evaluate
                    train_loss, train_acc, epoch_time = train_one_epoch(model, train_loader, criterion, optimizer, device)
                    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
                    
                    cumulative_time += epoch_time
                    
                    # Log the results for this epoch
                    results.append({
                        'Dataset': dataset_name,
                        'Model': ModelClass.__name__,
                        'Batch_Size': batch_size,
                        'Optimizer': opt_name,
                        'Epoch': epoch,
                        'Train_Loss': train_loss,
                        'Train_Acc': train_acc,
                        'Val_Loss': val_loss,
                        'Val_Acc': val_acc,
                        'Epoch_Time_sec': epoch_time,
                        'Cumulative_Time_sec': cumulative_time
                    })
                    
                    # Print progress every 5 epochs (except for the first epoch)
                    if epoch % 5 == 0 or epoch == 1:
                        print(f"  Epoch [{epoch}/{epochs}] | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    #  SAVE RESULTS 
    df_results = pd.DataFrame(results)
    df_results.to_csv("benchmark_experiment_results_transformer.csv", index=False)
    print("\nExperiment Complete. Results saved to 'benchmark_experiment_results_transformer.csv'")
    return df_results

In [ ]:
# Executing the transformer experiment
df = run_transformer_experiment()

In [8]:
# Set plotting style for academic presentation
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)

def load_and_clean_data(file_path="benchmark_experiment_results_transformer.csv"):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Cannot find {file_path}. Ensure the experiment has finished running.")
    
    df = pd.read_csv(file_path)
    
    # Handle NaN values explicitly (e.g., when SGD explodes)
    # Forward fill or replace with a recognizable string/value for tracking, 
    # but for plotting, matplotlib handles NaNs by breaking the line, which is perfect.
    
    return df

def plot_convergence_by_epoch(df, dataset_name, batch_size, save_dir="plots/vit"):
    subset = df[(df['Dataset'] == dataset_name) & (df['Batch_Size'] == batch_size)]
    if subset.empty:
        return
        
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"ViT Convergence: {dataset_name.upper()} | Batch Size: {batch_size}", fontsize=20, y=1.05)
    
    #  Plot 1: Train Loss vs Epoch 
    sns.lineplot(data=subset, x='Epoch', y='Train_Loss', hue='Optimizer', marker='o', ax=axes[0], linewidth=2.5)
    axes[0].set_title('Training Loss', fontsize=18)
    axes[0].set_ylabel('Cross Entropy Loss', fontsize=16)
    axes[0].set_xlabel('Epoch', fontsize=16)
    axes[0].legend(fontsize=14, title_fontsize=13)
    axes[0].tick_params(labelsize=14)
    
    valid_losses = subset['Train_Loss'].replace([np.inf, -np.inf], np.nan).dropna()
    if not valid_losses.empty:
        max_y = min(valid_losses.max(), valid_losses.quantile(0.95) * 1.5)
        axes[0].set_ylim(bottom=0, top=max(max_y, 2.5))
    
    #  Plot 2: Val Loss vs Epoch 
    sns.lineplot(data=subset, x='Epoch', y='Val_Loss', hue='Optimizer', marker='s', ax=axes[1], linewidth=2.5)
    axes[1].set_title('Validation Loss', fontsize=18)
    axes[1].set_ylabel('Loss', fontsize=16)
    axes[1].set_xlabel('Epoch', fontsize=16)
    axes[1].legend(fontsize=14, title_fontsize=13)
    axes[1].tick_params(labelsize=14)
    
    valid_val_losses = subset['Val_Loss'].replace([np.inf, -np.inf], np.nan).dropna()
    if not valid_val_losses.empty:
        max_y = min(valid_val_losses.max(), valid_val_losses.quantile(0.95) * 1.5)
        axes[1].set_ylim(bottom=0, top=max(max_y, 2.5))
    
    plt.tight_layout()
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(f"{save_dir}/{dataset_name}_vit_b{batch_size}_epochs.png", dpi=300, bbox_inches='tight')
    plt.close()

def plot_convergence_by_time(df, dataset_name, batch_size, save_dir="plots/vit"):
    """Plots Training Loss vs Cumulative Wall-clock Time (Grid Optimized)"""
    subset = df[(df['Dataset'] == dataset_name) & (df['Batch_Size'] == batch_size)]
    if subset.empty:
        return
        
    plt.figure(figsize=(6, 4))
    ax = sns.lineplot(
        data=subset, 
        x='Cumulative_Time_sec', 
        y='Train_Loss', 
        hue='Optimizer', 
        marker='X',
        linewidth=2.0,
        markersize=7,
        err_style="bars"
    )
    
    ax.set_title(
        f"Loss vs. Wall-Clock Time: {dataset_name.upper()} | Batch Size: {batch_size}",
        fontsize=13,
        pad=10
    )
    ax.set_xlabel('Cumulative Training Time (Seconds)', fontsize=11)
    ax.set_ylabel('Training Loss', fontsize=11)
    
    ax.tick_params(axis='both', which='major', labelsize=9)
    legend = ax.legend(fontsize=9)
    if legend is not None:
        plt.setp(legend.get_title(), fontsize=10)
    
    # Protect Y-axis from explosions
    valid_losses = subset['Train_Loss'].replace([np.inf, -np.inf], np.nan).dropna()
    if not valid_losses.empty:
        max_y = min(valid_losses.max(), valid_losses.quantile(0.95) * 1.5)
        plt.ylim(bottom=0, top=max(max_y, 2.5))
        
    plt.tight_layout()
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(
        f"{save_dir}/{dataset_name}_vit_b{batch_size}_time.png", 
        dpi=600, 
        bbox_inches='tight'
    )
    plt.close()

def plot_overhead_bar_chart(df, save_dir="plots/vit"):
    """Plots the average time taken per epoch for each optimizer"""
    # Group by Optimizer and Dataset to find the mean time per epoch
    avg_time = df.groupby(['Optimizer', 'Dataset'])['Epoch_Time_sec'].mean().reset_index()
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=avg_time, x='Dataset', y='Epoch_Time_sec', hue='Optimizer')
    
    plt.title("Computational Overhead: Average Seconds per Epoch (ViT)")
    plt.ylabel('Seconds per Epoch')
    plt.xlabel('Dataset / Model Architecture')
    
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(f"{save_dir}/vit_computational_overhead.png", dpi=300, bbox_inches='tight')
    plt.close()

def generate_transformer_visualizations(csv_path="benchmark_experiment_results_transformer.csv"):
    print("Loading experiment data...")
    try:
        df = load_and_clean_data(csv_path)
    except FileNotFoundError as e:
        print(e)
        return

    datasets = df['Dataset'].unique()
    batch_sizes = df['Batch_Size'].unique()
    
    print("Generating plots...")
    for dataset in datasets:
        for batch_size in batch_sizes:
            plot_convergence_by_epoch(df, dataset, batch_size)
            plot_convergence_by_time(df, dataset, batch_size)
            
    plot_overhead_bar_chart(df)
    print("All visualizations saved successfully in the 'plots/vit/' directory")

In [ ]:
# Transformer Visualization Execution
generate_transformer_visualizations()